In [10]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def scrape_ikman_houses(max_pages=2):
    all_data = []
    base_url = "https://ikman.lk/en/ads/sri-lanka/houses-for-sale"
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    }

    for page in range(1, max_pages + 1):
        print(f"Scraping page {page}...")
        
        # Construct URL for pagination
        if page == 1:
            url = base_url
        else:
            url = f"{base_url}?sort=date&order=desc&buy_now=0&urgent=0&page={page}"

        try:
            response = requests.get(url, headers=headers)
            if response.status_code != 200:
                print(f"Failed to retrieve page {page}. Status: {response.status_code}")
                break
                
            soup = BeautifulSoup(response.text, 'html.parser')
            # Identify all ad list items
            listings = soup.find_all('li', class_=lambda x: x and ('normal' in x or 'top-ads' in x))

            for ad in listings:
                try:
                    title = ad.find('h2').text.strip() if ad.find('h2') else "N/A"
                    
                    # Using lambda to find classes that "contain" the keyword (handles dynamic suffixes)
                    price = ad.find('div', class_=lambda x: x and 'price' in x).text.strip() if ad.find('div', class_=lambda x: x and 'price' in x) else "N/A"
                    details = ad.find('div', class_=lambda x: x and 'details' in x).text.strip() if ad.find('div', class_=lambda x: x and 'details' in x) else "N/A"
                    location = ad.find('div', class_=lambda x: x and 'description' in x).text.strip() if ad.find('div', class_=lambda x: x and 'description' in x) else "N/A"
                    
                    all_data.append({
                        "Title": title,
                        "Price": price,
                        "Details": details,
                        "Location": location,
                        "Page": page
                    })
                except Exception:
                    continue

            # Respectful delay to avoid getting IP banned
            time.sleep(2) 

        except Exception as e:
            print(f"An error occurred on page {page}: {e}")
            break

    # Convert list of dictionaries to a DataFrame
    df = pd.DataFrame(all_data)
    
    # Save to CSV
    df.to_csv("ikman_houses.csv", index=False, encoding='utf-8-sig')
    print(f"\nSuccess! Saved {len(df)} listings to 'ikman_houses.csv'")

# Run the scraper
scrape_ikman_houses(max_pages=200)

Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...
Scraping page 6...
Scraping page 7...
Scraping page 8...
Scraping page 9...
Scraping page 10...
Scraping page 11...
Scraping page 12...
Scraping page 13...
Scraping page 14...
Scraping page 15...
Scraping page 16...
Scraping page 17...
Scraping page 18...
Scraping page 19...
Scraping page 20...
Scraping page 21...
Scraping page 22...
Scraping page 23...
Scraping page 24...
Scraping page 25...
Scraping page 26...
Scraping page 27...
Scraping page 28...
Scraping page 29...
Scraping page 30...
Scraping page 31...
Scraping page 32...
Scraping page 33...
Scraping page 34...
Scraping page 35...
Scraping page 36...
Scraping page 37...
Scraping page 38...
Scraping page 39...
Scraping page 40...
Scraping page 41...
Scraping page 42...
Scraping page 43...
Scraping page 44...
Scraping page 45...
Scraping page 46...
Scraping page 47...
Scraping page 48...
Scraping page 49...
Scraping page 50...
Scraping 

In [22]:
import pandas as pd
import re

df = pd.read_csv("ikman_houses.csv")

In [ ]:
def extract_counts(text, label):
    match = re.search(fr'{label}: (\d+)', str(text))
    return int(match.group(1)) if match else 0

In [23]:
df['Price_Cleaned'] = df['Price'].str.replace(r'Rs|,', '', regex=True).str.strip().astype(float)
df['Details'] = df['Details'].fillna('Unknown')
df['Title'] = df['Title'].fillna('No Title')
df['Bedrooms'] = df['Details'].apply(lambda x: extract_counts(x, 'Bedrooms'))
df['Bathrooms'] = df['Details'].apply(lambda x: extract_counts(x, 'Bathrooms'))
df['City'] = df['Location'].str.split(',').str[0].str.strip()
df['Title_Cleaned'] = df['Title'].str.replace(r'\(.*?\)', '', regex=True).str.strip()

In [29]:
df_cleaned = df[['Title_Cleaned', 'Price_Cleaned', 'Bedrooms', 'Bathrooms', 'City']]
df_cleaned.rename(columns={'Title_Cleaned': 'Title', 'Price_Cleaned': 'Price (Rs.)'}, inplace=True)
df_cleaned.to_csv("ikman_houses_cleaned.csv", index=False)

C:\Users\user\AppData\Local\Temp\ipykernel_4504\710155843.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned.rename(columns={'Title_Cleaned': 'Title', 'Price_Cleaned': 'Price (Rs.)'}, inplace=True)


In [26]:
df_cleaned.head()

,Title_Cleaned,Price_Cleaned,Bedrooms,Bathrooms,City
0,Two Storied House for Sale In Rathgama,15500000.0,3,2,Galle
1,House For Sale Gampaha - Minuwangoda Yakahatuwa,6500000.0,1,1,Gampaha
2,House For Sale Gampaha - Minuwangoda Yakahatuwa,6500000.0,1,1,Gampaha
3,MODERN BRAND NEW HOUSE FOR SALE AT LONDON SQAU...,45000000.0,5,3,Colombo
4,Main Road Facing 52 P Sale At Pepiliyana-Belan...,286000000.0,1,1,Colombo
